In [ ]:
!pip install boto3
import os
import boto3
import pandas as pd
import glob
import json
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_score, recall_score, classification_report

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.9 MB/s eta 0:00:00


In [ ]:
LLM_MODEL = "us.anthropic.claude-3-5-haiku-20241022-v1:0"
client = boto3.client("bedrock-runtime", region_name="us-east-1")
os.environ['AWS_BEARER_TOKEN_BEDROCK'] = "<your_bedrock_token>"

try:
    demo_df = pd.read_csv("demo.csv")
    demo_df['SUB_ID'] = demo_df['SUB_ID'].astype(int)
    demo_df['ADM_ID_1'] = demo_df['ADM_ID_1'].astype(int)
except FileNotFoundError as e:
    print(f"Error: Required demographic file not found: {e}. Please ensure 'demo.csv' is present.")
    sys.exit(1)

VITALS_ROOT_DIR = '/content/Positive'
# VITALS_ROOT_DIR = '/content/Negative'
file_pattern = '**[0-9]*_[0-9]*.csv'
all_vitals_files = glob.glob(os.path.join(VITALS_ROOT_DIR, file_pattern), recursive=True)

all_results = []
print(f" Found {len(all_vitals_files)} vitals files to process across all subdirectories")

for vitals_path in all_vitals_files:
    base_name = os.path.basename(vitals_path).replace('.csv', '')
    try:
        subject_id, adm1_str = base_name.split('_')
        subject_id = int(subject_id)
        adm1 = int(adm1_str)
    except ValueError:
        print(f"Skipping: Filename format error in {vitals_path}. Expected SUBID_ADMID.csv")
        continue

    path_parts = vitals_path.split(os.sep)
    group = path_parts[-2] if len(path_parts) > 1 else 'Root'

    print(f"\n--- Processing Subject: {subject_id}, Admission: {adm1} (Folder: {group}) ---")
    row = demo_df[(demo_df.SUB_ID == subject_id) & (demo_df.ADM_ID_1 == adm1)]

    if row.empty:
        print(f"Skipping: Subject {subject_id} not found in the consolidated demo file.")
        continue

    actual_readmission_status = row.iloc[0]["ACTUAL_ADM"]
    print(actual_readmission_status)

    try:
        age = row.iloc[0]["AGE_AT_ADM_1"]
        gender = row.iloc[0]["GENDER"]

        ds1 = row.iloc[0]["DISCHARGE_NOTE_1"]
        if pd.isna(ds1) or len(str(ds1).strip()) == 0:
            ds1 = "NOT FOUND"
    except (IndexError, KeyError) as e:

        try:

             ds1 = row.iloc[0]["DISCHARGE_SUMMARY"]
             if pd.isna(ds1) or len(str(ds1).strip()) == 0:
                 ds1 = "NOT FOUND"
        except KeyError:
            print(f"Skipping: Error extracting demographics/discharge summary for {subject_id}: {e}")
            continue

    try:
        vitals_df = pd.read_csv(vitals_path)
        vitals_df['CHARTTIME_FLOAT'] = vitals_df['CHARTTIME'].astype(float)
        vitals_df['CHARTTIME_DT'] = pd.to_datetime(vitals_df['CHARTTIME_FLOAT'], unit='ns')
        vitals_text = vitals_df.to_string(index=False)
    except Exception as e:
        print(f"Skipping: Error reading or processing vitals file {vitals_path}: {e}")
        continue

    # PROMPT 1 (Vitals Only)

    prompt1 = f"""
        You are an AI clinical assistant. You will receive vital signs from a patient’
        s admission.
        Do NOT assume or infer anything about subsequent admissions or readmissions.
        Do NOT hallucinate any discharge summaries or clinical context.
        VITAL SIGNS (CSV): {vitals_text}
        PATIENT DEMOGRAPHICS: Age: {age}, Gender: {gender}
        ----------------------------------------------------
        TASKS
        ----------------------------------------------------
        1. Identify vital signs outside normal ranges appropriate for this patient’s
        age and gender.
        Apply clinically appropriate thresholds when interpreting abnormal values.
        2. Determine if the vitals show a worsening physiological trend (True/False).
        Consider only the vital-sign patterns. Do NOT use any clinical notes.
        ----------------------------------------------------
        STRICT OUTPUT FORMAT (JSON ONLY)
        ----------------------------------------------------
        {
        "abnormal_vitals": [
        {
        "vital": "...",
        "value": "...",
        "timestamp": "...",
        "reason": "..."
        }
        ],
        "worsening_trend": true/false,
        "clinical_context_flags": ["..."],
        "summary": "Short factual summary based ONLY on vitals."
        }
        """

    try:
        response1 = client.converse(
            modelId=LLM_MODEL,
            messages=[{"role": "user", "content": [{"text": prompt1}]}]
        )

        analysis_summary = response1["output"]["message"]["content"][0]["text"]
        cleaned_analysis_summary = analysis_summary.replace('\n', '').replace('\r', '')
        parsed_analysis = json.loads(cleaned_analysis_summary)
        analysis_summary_for_prompt2 = json.dumps(parsed_analysis, indent=2)

        print("\n[Vitals Analysis JSON (LLM1)]")
        print(analysis_summary_for_prompt2)

    except Exception as e:
        print(f"Skipping: LLM1 failed for {subject_id}_{adm1}. Error: {e}")
        continue

    # PROMPT 2 (LLM1 JSON + Discharge Summary)
    prompt2 = f"""
        You are a clinical risk - prediction model . Your ONLY inputs are:
        1. A structured JSON summary of FIRST ADMISSION vital signs (
        from LLM1 )
        2. A FIRST ADMISSION discharge summary note
        You MUST NOT assume , reference , or imagine any readmission or
        future information .
        Base your prediction ONLY on what clinicians would know at the
        time of discharge .
        RESPONSE FROM FIRST LLM: { analysis_summary }
        DISCHARGE SUMMARY ( FIRST ADMISSION ): {ds1 }
        PATIENT DEMOGRAPHICS : Age: {age}, Gender : { gender }
        ----------------------------------------------------
        TASKS
        ----------------------------------------------------
        1. Combine physiologic findings from the LLM1 JSON with objective findings in
        the discharge summary.
        2. Identify specific factors that increase 10-day readmission risk, including:
        - Hemodynamic instability
        - Respiratory compromise
        - Infection/inflammation
        - Renal impairment
        - Heart failure instability
        - Functional decline
        - Medication issues
        3. Identify protective factors that decrease readmission likelihood, such as:
        - Stable vitals
        - Resolved infection
        - Functional stability
        - Arranged follow-up care
        4. Evaluate how the discharge summary interacts with the vital-sign data.
        ----------------------------------------------------
        STRICT OUTPUT FORMAT (JSON ONLY)
        ----------------------------------------------------
        {
        "risk_factors": [ "factor1", "factor2" ],
        "protective_factors": [ "factor1", "factor2" ],
        "readmission_risk_percent": 0,
        "rationale": "A detailed, evidence-based justification according to the rules above."
        }
        """

    # LLM2 CALL AND DATA PARSING
    try:
        response2 = client.converse(
            modelId=LLM_MODEL,
            messages=[{"role": "user", "content": [{"text": prompt2}]}]
        )
        final_prediction = response2["output"]["message"]["content"][0]["text"]

        cleaned_prediction = final_prediction.replace('\n', '').replace('\r', '')
        result_json = json.loads(cleaned_prediction)

        # ADD IDENTIFIERS AND ACTUAL OUTCOME
        result_json['subject_id'] = subject_id
        result_json['adm_id'] = adm1
        result_json['actual_readmission'] = actual_readmission_status # GT derived from ADM_ID_2 presence

        # DERIVE PREDICTED READMISSION
        risk_percent = result_json.get('readmission_risk_percent', 0)
        result_json['pred_readmission'] = 'Yes' if risk_percent >= 50 else 'No'

        # APPEND THE ENTIRE DICTIONARY TO THE MASTER LIST
        all_results.append(result_json)

        print("\n[Final Prediction JSON (LLM2)]")
        print(json.dumps(result_json, indent=2))

    except json.JSONDecodeError as e:
        print(f"**ERROR:** Failed to parse FINAL JSON for {subject_id}_{adm1}. Decode Error: {e}")
        continue

    except Exception as e:
        print(f"Skipping: LLM2 call failed for {subject_id}: {e}")
        continue


print("\n=======================================================")
print(f"PROCESSING COMPLETE. Total {len(all_results)} subjects processed.")
print("=======================================================")

if all_results:

    final_df = pd.DataFrame(all_results)

    desired_columns = [
        'subject_id',
        'adm_id',
        'actual_readmission',
        'pred_readmission',
        'readmission_risk_percent',
        'rationale',
    ]


    final_df_selected = final_df.reindex(columns=desired_columns)

    output_filename = "readmission_predictions_final.csv"
    final_df_selected.to_csv(output_filename, index=False)

    print(f"✅ Results saved successfully to **'{output_filename}'**")

--- Found 26 vitals files to process across all subdirectories ---

--- Processing Subject: 72482, Admission: 161011 (Folder: Positive) ---
Yes

[Vitals Analysis JSON (LLM1)]
{
  "abnormal_vitals": [
    {
      "vital": "Heart Rate",
      "value": "Frequently below 60 bpm (bradycardia)",
      "timestamp": "Multiple instances, e.g., 2123-09-15 16:00:00",
      "reason": "HR drops to 60-65 bpm range multiple times"
    },
    {
      "vital": "Respiratory Rate",
      "value": "Frequently outside 12-20 breaths/minute",
      "timestamp": "Multiple instances, e.g., 2123-09-11 08:00:00",
      "reason": "RR often elevated to 23-27 breaths/minute"
    },
    {
      "vital": "Blood Pressure",
      "value": "Systolic BP frequently elevated",
      "timestamp": "2123-09-08 12:00:00",
      "reason": "Systolic BP reaches up to 202 mmHg, well above normal range"
    },
    {
      "vital": "Temperature",
      "value": "Transient fever",
      "timestamp": "2123-09-19 00:00:00",
      "reas